# データベース演習 第14回

### 🔍 利用方法

- このノートブックは**閲覧専用**です。  
  自分のGoogleドライブにコピーを作成してから編集してください。

  1. メニューの「**ファイル → ドライブにコピーを保存**」を選ぶ  
  2. コピーしたノートブックの**ファイル名（左上の"xxxx"）を学籍番号に変更**してください（提出時の識別のため）

---

### 🔧 提出手順

1. Google Colab の右上にある「**共有**」ボタンをクリック  
2. 「**一般的なアクセス**」の設定を「**リンクを知っている全員**」に変更  
3. アクセス権を「**閲覧者**」に設定（⚠️「編集者」にしないこと！）  
4. 表示されるURLをコピー  
5. WebClassの提出フォームに、その**URLを貼り付けて提出**

---

### 💡 補足・注意点

- **「編集者」ではなく「閲覧者」**に設定してください。  
  → 教員が間違ってファイルを上書きしてしまうのを防ぐためです。
- **提出したリンクを自分でも一度開いてみて、ちゃんと共有されているかを確認**してください。

### データベースをダウンロード

* 全データが含まれています
* Colabのランタイムが切り替わると，その度にダウンロードする必要があります
* その度にダウンロードするのが面倒な人は，自分のGoogleドライブに保存し，そのファイルを参照する方法もあります（生成AIなどで調べてみてください）．

In [ ]:
# SQLiteファイルをダウンロード

# ex_finalデータベース
!curl -sLO https://raw.githubusercontent.com/ggszk/ggszk-lab-public/refs/heads/main/db/ex_final.sqlite3


### JupySQLのインストールと有効化

In [ ]:
# Colabではjupysqlのインストールが必要
import sys
if 'google.colab' in sys.modules:
    %pip install -q jupysql

In [ ]:
# jupysqlの拡張機能を有効化
%load_ext sql

In [ ]:
# 結果表示数は以下の数値を変えれば変更できる
%config SqlMagic.displaylimit = 100 # デフォルトは10行

### ex_finalデータベース

In [ ]:
# ex_finalデータベースに接続する
%sql sqlite:///ex_final.sqlite3

### 含まれるテーブルの確認

In [ ]:
%%sql
SELECT * FROM sqlite_master;


## 例：主キーの調査（メタデータ）

SQLiteでは，PRAGMA関数を使ってテーブルのメタデータをSQLで調べられる．
pk列が1以上なら主キーである（複合主キーの場合は主キー内の順番を表す）．
notnull列が1ならNOT NULL制約が設定されている．


In [ ]:
%%sql
SELECT name, "notnull", pk FROM pragma_table_info('order_details');

## 例：外部キーの検証

**ex_finalデータベースには外部キー制約が定義されていない**（古いシステムでは珍しくない）．
よって外部キーはメタデータからは読み取れない．次の手順で調査する．

1. カラム名・仕様書から「このカラムは外部キーではないか」という仮説を立てる
2. そのカラムの値がすべて参照先の主キーに存在するかをSQLで調べる（結果が0件なら仮説が支持される）

例：order_detailsのorder_idは，ordersの主キーと同名なので外部キーと予想できる．検証する．


In [ ]:
%%sql
-- order_details.order_idの値のうち，ordersの主キーに存在しないものの個数（0件なら外部キーの仮説が支持される）
SELECT COUNT(*) FROM order_details
  WHERE order_id NOT IN (SELECT order_id FROM orders);

## 例題1

territoriesテーブル（テリトリー：従業員の担当範囲）とemployee_territoriesテーブル（従業員とテリトリーの関連を表すテーブル）について，主キー・外部キーを調査せよ

* 主キー：pragma_table_infoで調査する
* 外部キー：カラム名から仮説を立て，SQLで検証する
* 調査結果を下の表に記入すること（調査に使ったSQLのセルも残すこと）

| テーブル | 主キー | 外部キー（参照先） |
|---------|-------|------------------|
| territories |  |  |
| employee_territories |  |  |

この関連の多重度（1対多のどちらが多側＝矢印側か）：


In [ ]:
%%sql


In [ ]:
%%sql


## 例：SQLによるオプショナリティの調査

例：employee_territoriesとterritoriesの関連のオプショナリティを現在のデータを参考にして決定せよ

### territories側のオプショナリティの調査


In [ ]:
%%sql
SELECT COUNT(*) FROM employee_territories WHERE territory_id IS null;

### employee_territories側のオプショナリティの調査

In [ ]:
%%sql
SELECT COUNT(territory_id) FROM territories
  WHERE territory_id NOT IN (
    SELECT territory_id FROM employee_territories
    WHERE territory_id IS NOT null
);

## 例題2

customersとordersの間の関連について，オプショナリティを決定せよ

1. customers側のオプショナリティを調べるためのSQL文を作成せよ


In [ ]:
%%sql


2. orders側のオプショナリティを調べるためのSQL文を作成せよ

In [ ]:
%%sql


## 以下，最終課題

問1〜問3・問7はこのノートブック上で解答する．提出は，これまでの演習と同様にWebClassの最終課題にColabの共有リンクを提出する．

### 最終課題 問1

以下の5つのテーブルについて主キー・外部キーを調査し，各テーブルの記入表に結果を記録せよ

* 対象：orders，order_details，products，employees，employee_territories
* 主キー：pragma_table_infoで調査する（実行したセルを残すこと）
* 外部キー：カラム名・仕様書から仮説を立て，SQLで検証する（検証に使ったSQLと実行結果のセルを残すこと）
* ヒント：複合主キーのテーブル，自分自身を参照する外部キーを持つテーブルが含まれている


#### ordersテーブル

| カラム名 | 主キー（○と順番） | 外部キー（参照先テーブル.カラム） | 根拠メモ |
|---------|------------------|--------------------------------|----------|
| order_id |  |  |  |
| customer_id |  |  |  |
| employee_id |  |  |  |
| order_date |  |  |  |
| required_date |  |  |  |
| shipped_date |  |  |  |
| ship_via |  |  |  |
| freight |  |  |  |
| ship_name |  |  |  |
| ship_address |  |  |  |
| ship_city |  |  |  |
| ship_region |  |  |  |
| ship_postal_code |  |  |  |
| ship_country |  |  |  |


In [ ]:
%%sql


In [ ]:
%%sql


#### order_detailsテーブル

| カラム名 | 主キー（○と順番） | 外部キー（参照先テーブル.カラム） | 根拠メモ |
|---------|------------------|--------------------------------|----------|
| order_id |  |  |  |
| product_id |  |  |  |
| unit_price |  |  |  |
| quantity |  |  |  |
| discount |  |  |  |


In [ ]:
%%sql


In [ ]:
%%sql


#### productsテーブル

| カラム名 | 主キー（○と順番） | 外部キー（参照先テーブル.カラム） | 根拠メモ |
|---------|------------------|--------------------------------|----------|
| product_id |  |  |  |
| product_name |  |  |  |
| supplier_id |  |  |  |
| category_id |  |  |  |
| quantity_per_unit |  |  |  |
| unit_price |  |  |  |
| units_in_stock |  |  |  |
| units_on_order |  |  |  |
| reorder_level |  |  |  |
| discontinued |  |  |  |


In [ ]:
%%sql


In [ ]:
%%sql


#### employeesテーブル

| カラム名 | 主キー（○と順番） | 外部キー（参照先テーブル.カラム） | 根拠メモ |
|---------|------------------|--------------------------------|----------|
| employee_id |  |  |  |
| last_name |  |  |  |
| first_name |  |  |  |
| title |  |  |  |
| title_of_courtesy |  |  |  |
| birth_date |  |  |  |
| hire_date |  |  |  |
| address |  |  |  |
| city |  |  |  |
| region |  |  |  |
| postal_code |  |  |  |
| country |  |  |  |
| home_phone |  |  |  |
| extension |  |  |  |
| photo |  |  |  |
| notes |  |  |  |
| reports_to |  |  |  |
| photo_path |  |  |  |


In [ ]:
%%sql


In [ ]:
%%sql


#### employee_territoriesテーブル

| カラム名 | 主キー（○と順番） | 外部キー（参照先テーブル.カラム） | 根拠メモ |
|---------|------------------|--------------------------------|----------|
| employee_id |  |  |  |
| territory_id |  |  |  |


In [ ]:
%%sql


In [ ]:
%%sql


### 最終課題 問2

ex_finalデータベースの関連（customer_customer_demoとcustomer_demographicsは除く）について，多重度とオプショナリティを調査し，下の表に記録して，ER図を完成させよ

* 1対多の向き（どちらが多側か）と，両側のオプショナリティ（黒丸/白丸）をそれぞれ記録すること
* 各行に判断の根拠を必ず記入すること（仕様書の記述番号・常識的判断・調査SQLのいずれか）
* SQLで判断した箇所は，そのSQLと実行結果のセルがこの下に残っていること
* ネット上の資料や生成AIを参考にすることは禁止しないが，このデータベースのデータに基づく根拠がない解答は得点にならない

| 1側テーブル | 多側テーブル（外部キー） | 1側の黒丸/白丸 | 多側の黒丸/白丸 | 根拠 |
|-----------|------------------------|--------------|---------------|------|
| orders（記入例） | order_details (order_id) | 黒丸 | 黒丸 | 仕様書1・常識 |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |


問2の調査用セル（必要なだけ追加してよい）

In [ ]:
%%sql


In [ ]:
%%sql


In [ ]:
%%sql


In [ ]:
%%sql


### 最終課題 問3

オプショナリティの調査で説明したSQLを参考にして，地域テーブルとテリトリテーブルの関連のオプショナリティを調査するため，以下の2つのSQL文を作成せよ

(1) テリトリテーブルのregion_idがnullである行数を求めるSQL文

In [ ]:
%%sql


(2) テリトリテーブルのregion_idカラムには存在せず，地域テーブルのregion_idカラムに存在するregion_idの値の個数を求めるSQL文

In [ ]:
%%sql


従業員テーブルには，従業員の報告先（上司）を表すカラムがある（報告先従業員のIDが入る）．このカラムに関して，以下の2つのSQL文を作成せよ（このSQL文はオプショナリティのヒントになっている）

(3) 報告先（上司）が存在しない従業員の人数を求めるSQL文

In [ ]:
%%sql


(4) 部下（自分を報告先としている人）を持っている従業員の人数を求めるSQL文

In [ ]:
%%sql


### 最終課題 問7（発展問題）：データベースの拡張設計

ex_finalに新しい機能を追加することになった．以下の要件から**1つ選び**，そのために必要なテーブルを設計せよ

* **要件A（商品レビュー）**：顧客が製品にレビュー（5段階の評価と本文）を投稿できるようにしたい
* **要件B（配送状況の追跡）**：注文ごとの配送状況の変化（発送済・配達中・配達完了など）を，日時とともに履歴として記録したい
* **要件C（複数倉庫の在庫管理）**：製品の在庫数を，複数の倉庫ごとに管理したい

解答として以下を記述すること

1. 追加するテーブルの定義（テーブル名・カラム名・主キー・外部キー）：下の記入表に記述
2. 追加テーブルと既存テーブルとの関連を含むER図：Mermaid記法で記述し，show_er()で描画
3. 設計理由の説明：なぜその主キー・外部キーにしたか，多重度・オプショナリティをどう考えたか

注意事項

* この問題には唯一の正解はない．採点では最終的な図の見た目よりも，**設計の理由が筋の通った言葉で説明されていること**を重視する
* 生成AIに相談してもよいが，**最初の案は必ず自分で考えて書くこと**．AIの指摘を採用/不採用した場合は，その理由も記述すること

Mermaid記法の記号（授業の記法との対応）

| 授業の記法 | Mermaid | 意味 |
|-----------|---------|------|
| 黒丸（必須）・1側 | `\|\|` | ちょうど1 |
| 白丸（オプション）・1側 | `\|o` | 0か1 |
| 黒丸（必須）・多側 | `\|{` | 1以上 |
| 白丸（オプション）・多側 | `o{` | 0以上 |


In [ ]:
# Mermaid記法のER図を描画するヘルパー関数（実行しておくこと）
from IPython.display import Image, display
import base64

def show_er(mermaid_code):
    """Mermaidコードを画像として表示する"""
    encoded = base64.urlsafe_b64encode(mermaid_code.encode()).decode()
    display(Image(url=f"https://mermaid.ink/img/{encoded}"))

In [ ]:
# 記述例：この3行の意味を確認しよう（実行すると図が表示される）
# 注意：描画される図は，授業で使った黒丸・白丸や矢印ではなく，
# 線の端の形（カラスの足）で多重度・オプショナリティを表す．上の対応表と見比べること
show_er("""
erDiagram
    customers ||--o{ orders : ""
    orders ||--|{ order_details : ""
    products ||--o{ order_details : ""
""")

#### 解答欄1：追加するテーブルの定義

| テーブル名 | カラム名 | 主キー | 外部キー（参照先） | 意味 |
|-----------|---------|-------|------------------|------|
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |


In [ ]:
# 解答欄2：ER図（Mermaid記法で記述して実行する）
show_er("""
erDiagram
""")

#### 解答欄3：設計理由

* 主キーの選択の理由：
* 外部キー・関連（多重度・オプショナリティ）の考え方：
* （生成AIを利用した場合）指摘を採用/不採用した理由：
